In [ ]:
# Lecture 14 — DataFrames & Schema
# Spark 101 · Module 5 · Lecture 14
#
# Import this file into a Databricks workspace as a notebook.
# Databricks recognizes `# COMMAND ----------` as a cell boundary on import.
# In VS Code / PyCharm this stays a plain Python file — same source, two homes.

In [ ]:
print("Spark version:", spark.version)
# Databricks gives you `spark` for free — no SparkSession.builder needed.

In [ ]:
# --- A DataFrame from a Python collection ---------------------------------
# Spark infers the schema from the Python objects: str -> string, int -> long.
people = [
    ("Ada",   36, "Madrid"),
    ("Linus", 29, "Helsinki"),
    ("Grace", 41, "New York"),
]
df = spark.createDataFrame(people, ["name", "age", "city"])
df.show()
df.printSchema()
# printSchema is your first move on ANY new DataFrame — look at the shape
# before you look at the rows.

In [ ]:
# --- Stage a small CSV so we can watch schema INFERENCE happen -------------
# We write it to the driver's local disk and read it back with the `file:`
# prefix — no upload, no Volume, no cloud storage needed for the demo.
csv_text = """id,zip,amount,signup_ts
1,08001,19.99,2026-01-15
2,28013,5.50,2026-02-03
3,00501,1200.00,2026-03-21
"""
path = "/tmp/spark101_l14_customers.csv"
with open(path, "w") as f:
    f.write(csv_text)
print("wrote", path)

In [ ]:
# --- Read #1: no inference. Every column comes back as a string. ----------
df_str = spark.read.csv("file:" + path, header=True)
df_str.printSchema()
# Notice: id, amount, signup_ts — all `string`. Spark did not guess anything.

In [ ]:
# --- Read #2: inferSchema=True. Spark guesses the types. ------------------
# Convenient — but it costs an extra pass over the data to sample it,
# and the guesses can be wrong.
df_inferred = spark.read.csv("file:" + path, header=True, inferSchema=True)
df_inferred.printSchema()
df_inferred.show()
# The trap: `zip` was 08001 / 00501 — leading zeros. Inference reads it as an
# integer and the leading zero VANISHES. A guess that silently corrupts data.

In [ ]:
# --- Read #3: explicit schema. You declare the shape up front. ------------
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType, DateType,
)

schema = StructType([
    StructField("id",        IntegerType(), False),
    StructField("zip",       StringType(),  True),   # keep as string -> zeros survive
    StructField("amount",    DoubleType(),  True),
    StructField("signup_ts", DateType(),    True),
])

df_explicit = spark.read.csv("file:" + path, header=True, schema=schema)
df_explicit.printSchema()
df_explicit.show()
# No extra scan happened — you told Spark the shape and it believed you.
# `zip` is a string now, so 08001 stays 08001.

In [ ]:
# --- Why types matter: string "+" is concatenation, not arithmetic --------
from pyspark.sql.functions import col

# amount-as-string glued together vs amount-as-double added together
as_string = df_str.select((col("amount") + col("amount")).alias("string_plus"))
as_double = df_explicit.select((col("amount") + col("amount")).alias("double_plus"))
as_string.show()   # nulls / wrong — Spark can't add strings the way you meant
as_double.show()    # 39.98, 11.0, 2400.0 — real arithmetic

In [ ]:
# --- The inspection toolkit -----------------------------------------------
df_explicit.printSchema()          # names, types, nullability (tree)
df_explicit.show(2)                # first n rows, formatted — SAFE, not a collect
print(df_explicit.columns)         # ['id', 'zip', 'amount', 'signup_ts']
print(df_explicit.dtypes)          # [('id', 'int'), ('zip', 'string'), ...]

In [ ]:
# withColumn: derive a new column from existing ones.
from pyspark.sql.functions import col, when
df.withColumn("bracket", when(col("age") >= 40, "senior").otherwise("junior")).show()

In [ ]:
# Rename and drop columns.
df.withColumnRenamed("name", "full_name").drop("city").show()

In [ ]:
# describe() / summary — quick stats on columns.
df.describe("age").show()
df_explicit.summary("count", "min", "max").show()

In [ ]:
# selectExpr: SQL-style expressions inside select.
df.selectExpr("name", "age", "age + 10 AS age_in_10y").show()

In [ ]:
# Recap:
#   - A DataFrame = a distributed table with named, typed columns + a schema.
#   - Infer while exploring; declare an explicit StructType when it matters.
#   - printSchema + show are your two constant companions.
# Next lecture: reading real data — CSV, JSON, Parquet.